# Quick creation test — runs on Colab **or** locally

One command creates handwriting from text. The engine is auto-picked:
**One-DM on a CUDA GPU** (best fidelity) or **HWT everywhere else** (CPU included).
No handwriting photo? The bundled handwriting samples are used, so this always runs.

Runs the same code a GPU PC runs — Colab just clones this repo first.

## 0. Where am I running?

In [ ]:
import os, sys, pathlib
IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content/.config")
print("Running on:", "Google Colab" if IN_COLAB else "local machine (GPU PC or CPU laptop)")

## 1. Get the project code here

In [ ]:
REPO_URL = "https://github.com/Jeevant010/Text_Writter"
if IN_COLAB:
    root = pathlib.Path("/content/Text_Writter")
    if not (root / "src/textwritter/quicktest.py").exists():
        !git clone --depth 1 {REPO_URL} {root}
    os.chdir(root)
else:
    here = pathlib.Path.cwd()
    root = here.parent if here.name == "notebooks" else here
    os.chdir(root)
print("repo root:", pathlib.Path.cwd())

## 2. Dependencies (skipped if already importable)

In [ ]:
missing = []
for mod in ["torch", "torchvision", "cv2", "PIL", "transformers", "numpy", "matplotlib", "gdown"]:
    try:
        __import__(mod)
    except Exception:
        missing.append(mod)
if missing:
    print("installing:", missing)
    !{sys.executable} -m pip install -q -r requirements.txt
else:
    print("dependencies already available")

## 3. Your handwriting (optional)

Upload one photo of a written line/paragraph — or leave it blank and the bundled
samples are used.

In [ ]:
STYLE = None
if IN_COLAB:
    from google.colab import files
    print("Upload ONE photo of your handwriting (blank = bundled samples)")
    up = files.upload()
    if up:
        name = list(up)[0]
        pathlib.Path("samples").mkdir(exist_ok=True)
        pathlib.Path(name).rename(root / "samples" / name)
        STYLE = str(root / "samples" / name)
else:
    import glob
    cands = [p for p in sorted(glob.glob("samples/*.jpg") + glob.glob("samples/*.png")
                               + glob.glob("samples/*.jpeg")) if "sample_hello" not in p]
    STYLE = cands[0] if cands else None
print("style:", STYLE or "none -> bundled handwriting samples")

## 4. Settings

In [ ]:
ENGINE = "auto"       # "auto" | "hwt" | "onedm"   (auto = One-DM on CUDA, else HWT)
TARGET_TEXT = "The quick brown fox jumps over the lazy dog"
STEPS = 50            # One-DM sampling steps (ignored by HWT)

## 5. Create

In [ ]:
import subprocess
cmd = [sys.executable, "src/textwritter/quicktest.py", "--engine", ENGINE,
       "--text", TARGET_TEXT, "--steps", str(STEPS)]
if STYLE:
    cmd += ["--style", STYLE]
print(" ".join(cmd))
print("exit:", subprocess.run(cmd).returncode)

## 6. Result (generated image + style comparison)

In [ ]:
import glob
from IPython.display import Image as IImage, display
outs = [p for p in sorted(glob.glob("out/quicktest_*.png"), key=os.path.getmtime)
        if not p.endswith("_compare.png")]
if outs:
    display(IImage(filename=outs[-1], width=900))
    cmp_ = outs[-1].replace(".png", "_compare.png")
    if pathlib.Path(cmp_).exists():
        print("top = style sample, bottom = generated")
        display(IImage(filename=cmp_, width=900))
else:
    print("no output found — read the log above")

## What next?

- Want a different look? Re-run with a real handwriting photo in `STYLE`.
- Force an engine: set `ENGINE = "hwt"` (fast, CPU-safe) or `"onedm"` (GPU).
- More text: edit `TARGET_TEXT`.
- Fine-tune for a closer match (optional, GPU only): see `01` / `02` notebooks.